# Stage 06 - Create the Integrated Test Ontology

Bind the ontology package to the authoritative Stage 03 Lakehouse products and Stage 04 Eventhouse tables.

In [ ]:
import importlib.util

if importlib.util.find_spec('fabricontology') is None:
    raise RuntimeError(
        'Attach a Fabric Environment containing the checked-in fabricontology wheel before running this notebook.'
    )

In [ ]:
import json

import sempy.fabric as fabric
from fabricontology import create_ontology_item, generate_definition_from_package
from notebookutils import mssparkutils

workspace_id = fabric.get_workspace_id()
access_token = mssparkutils.credentials.getToken('pbi')
ontology_package_path = '/lakehouse/default/Files/fabric-demos/06-ai-data-agent/Ontology/mda_test_ontology.iq'

ontology_item_name = 'mda_test_ontology'
binding_lakehouse_name = 'IntegratedTestLakehouse'
binding_lakehouse_schema_name = 'dbo'
binding_eventhouse_name = 'eh_mda_test'
binding_eventhouse_cluster_uri = '<eventhouse query uri>'
binding_eventhouse_database_name = 'kqldb_mda_test'

if binding_eventhouse_cluster_uri.startswith('<'):
    raise ValueError('Set binding_eventhouse_cluster_uri to the Stage 04 Eventhouse query URI before execution.')

items_df = fabric.list_items()
lakehouse_item = items_df[(items_df['Type'] == 'Lakehouse') & (items_df['Display Name'] == binding_lakehouse_name)]
eventhouse_item = items_df[(items_df['Type'] == 'Eventhouse') & (items_df['Display Name'] == binding_eventhouse_name)]
if lakehouse_item.empty:
    raise ValueError(f'Lakehouse not found: {binding_lakehouse_name}')
if eventhouse_item.empty:
    raise ValueError(f'Eventhouse not found: {binding_eventhouse_name}')

ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = generate_definition_from_package(
    ontology_package_path=ontology_package_path,
    ontology_name=ontology_item_name,
    binding_workspace_id=workspace_id,
    binding_lakehouse_item_id=str(lakehouse_item.iloc[0].Id),
    binding_lakehouse_schema_name=binding_lakehouse_schema_name,
    binding_eventhouse_item_id=str(eventhouse_item.iloc[0].Id),
    binding_eventhouse_cluster_uri=binding_eventhouse_cluster_uri,
    binding_eventhouse_database_name=binding_eventhouse_database_name,
)

response = create_ontology_item(
    workspace_id=workspace_id,
    access_token=access_token,
    ontology_item_name=ontology_item_name,
    ontology_definition=ontology_definition,
)
print(json.dumps(response.json(), indent=2))